# 01 Train Teachers

Trains frozen ImageNet teacher encoders with fresh linear classifier heads for all Phase 1 baselines. Each completed run writes the exact per-epoch metrics used by its training-curve figure to CSV.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

from src.teachers import TeacherModel
from src.train import evaluate, train_teacher_classifier, train_teacher_finetune
from src.utils import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    compute_flops,
    count_parameters,
    count_total_parameters,
    get_dataloaders,
    plot_training_curves,
    save_training_history_csv,
    set_seed,
)

set_seed(291652)

DATA_ROOT = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints" / "teachers"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
for directory in (
    CHECKPOINT_DIR / "frozen",
    CHECKPOINT_DIR / "finetune",
    FIGURE_DIR,
    TABLE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

In [ ]:
TEACHERS = ["convnext_tiny"]
DATASETS = ["cifar-100", "flowers-102", "tiny-imagenet-200"]
NUM_WORKERS = 6

DEFAULTS = dict(num_epochs=2, lr=1e-3, weight_decay=1e-4, batch_size=256)

RUN_OVERRIDES = {
    ("convnext_tiny", "flowers-102"):   dict(num_epochs=2, batch_size=64),
}

In [3]:
def checkpoint_path(teacher_name: str, dataset_name: str, mode: str = "frozen") -> Path:
    """Returns the checkpoint path for a (teacher, dataset, mode) triple.

    Args:
        teacher_name: Backbone identifier.
        dataset_name: Dataset identifier.
        mode: ``"frozen"`` for linear-probe checkpoints, ``"finetune"`` for
            partially-unfrozen encoder checkpoints.
    """
    return CHECKPOINT_DIR / mode / f"{teacher_name}_{dataset_name}.pth"


def history_csv_path(teacher_name: str, dataset_name: str, mode: str = "frozen") -> Path:
    return TABLE_DIR / f"training_history_{mode}_{teacher_name}_{dataset_name}.csv"


def describe_model(model: TeacherModel, loader) -> None:
    model.to(DEVICE)
    model.eval()
    images, _ = next(iter(loader))
    with torch.no_grad():
        feature_map, pooled = model.extract_features(images[:1].to(DEVICE))
    print(f"Backbone: {model.backbone_name}")
    print(f"Total params: {count_total_parameters(model) / 1e6:.2f}M")
    print(f"Trainable params: {count_parameters(model):,}")
    print(f"Encoder output shape: {tuple(feature_map.shape)}")
    print(f"Feature dim: {model.feature_dim} | pooled shape: {tuple(pooled.shape)}")

In [ ]:
histories = {}
history_frames = []

for teacher_name in TEACHERS:
    for dataset_name in DATASETS:
        cfg = {**DEFAULTS, **RUN_OVERRIDES.get((teacher_name, dataset_name), {})}

        print("=" * 100)
        print(f"Teacher: {teacher_name} | Dataset: {dataset_name} | cfg: {cfg}")
        set_seed(291652)

        train_loader, val_loader, _, num_classes = get_dataloaders(
            dataset_name=dataset_name,
            data_root=DATA_ROOT,
            batch_size=cfg["batch_size"],
            num_workers=NUM_WORKERS,
        )
        model = TeacherModel(backbone_name=teacher_name, num_classes=num_classes)
        describe_model(model, train_loader)

        history = train_teacher_classifier(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=cfg["num_epochs"],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
            device=DEVICE,
            checkpoint_path=str(checkpoint_path(teacher_name, dataset_name, mode="frozen")),
        )
        histories[(teacher_name, dataset_name)] = history

        history_frame = save_training_history_csv(
            history=history,
            save_path=history_csv_path(teacher_name, dataset_name, mode="frozen"),
            teacher_name=teacher_name,
            dataset_name=dataset_name,
        )
        history_frames.append(history_frame)
        pd.concat(history_frames, ignore_index=True).to_csv(
            TABLE_DIR / "teacher_training_histories_frozen.csv",
            index=False,
        )

        plot_training_curves(
            history=history,
            title=f"{teacher_name} on {dataset_name} (frozen)",
            save_path=FIGURE_DIR / f"teacher_frozen_{teacher_name}_{dataset_name}.png",
        )
        plt.show()
        del model; torch.cuda.empty_cache()

In [ ]:
results = []
flop_cache = {}

for teacher_name in TEACHERS:
    for dataset_name in DATASETS:
        saved_checkpoint = checkpoint_path(teacher_name, dataset_name, mode="frozen")
        if not saved_checkpoint.exists():
            print(f"Skipping missing checkpoint: {saved_checkpoint}")
            continue

        _, _, test_loader, num_classes = get_dataloaders(
            dataset_name=dataset_name,
            data_root=DATA_ROOT,
            batch_size=DEFAULTS["batch_size"],
            num_workers=NUM_WORKERS,
        )
        model = TeacherModel(
            backbone_name=teacher_name,
            num_classes=num_classes,
        ).to(DEVICE)
        checkpoint = torch.load(saved_checkpoint, map_location=DEVICE)
        model.classifier.load_state_dict(checkpoint["classifier_state_dict"])

        _, test_acc = evaluate(model, test_loader, DEVICE)
        flop_key = (teacher_name, num_classes)
        if flop_key not in flop_cache:
            try:
                flop_cache[flop_key] = compute_flops(model)
            except Exception as exc:
                print(f"Could not compute FLOPs for {teacher_name}/{dataset_name}: {exc}")
                flop_cache[flop_key] = np.nan

        results.append(
            {
                "Teacher": teacher_name,
                "Dataset": dataset_name,
                "Val Acc (%)": checkpoint.get("best_val_acc", np.nan),
                "Test Acc (%)": test_acc,
                "Total Params (M)": count_total_parameters(model) / 1e6,
                "Trainable Params": count_parameters(model),
                "GFLOPs": flop_cache[flop_key],
            }
        )
        del model; torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(TABLE_DIR / "teacher_results.csv", index=False)

In [ ]:
def unnormalize(image: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN, dtype=image.dtype).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, dtype=image.dtype).view(3, 1, 1)
    image = (image.cpu() * std + mean).clamp(0, 1)
    return image.permute(1, 2, 0).numpy()


for dataset_name in DATASETS:
    _, _, test_loader, num_classes = get_dataloaders(
        dataset_name=dataset_name,
        data_root=DATA_ROOT,
        batch_size=1,
        num_workers=NUM_WORKERS,
    )
    image, _ = next(iter(test_loader))
    figure, axes = plt.subplots(1, len(TEACHERS), figsize=(5 * len(TEACHERS), 4))
    axes = np.atleast_1d(axes)

    for axis, teacher_name in zip(axes, TEACHERS):
        saved_checkpoint = checkpoint_path(teacher_name, dataset_name, mode="frozen")
        if not saved_checkpoint.exists():
            axis.axis("off")
            axis.set_title(f"{teacher_name}\nmissing checkpoint")
            continue

        model = TeacherModel(
            backbone_name=teacher_name,
            num_classes=num_classes,
        ).to(DEVICE)
        checkpoint = torch.load(saved_checkpoint, map_location=DEVICE)
        model.classifier.load_state_dict(checkpoint["classifier_state_dict"])
        model.eval()

        with torch.no_grad():
            feature_map, _ = model.extract_features(image.to(DEVICE))
        activation = feature_map.mean(dim=1, keepdim=True)
        activation = F.interpolate(
            activation,
            size=image.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).squeeze().float().cpu()
        activation = (activation - activation.min()) / (activation.max() - activation.min() + 1e-8)

        axis.imshow(unnormalize(image[0]))
        axis.imshow(activation.numpy(), cmap="magma", alpha=0.45)
        axis.set_title(teacher_name)
        axis.axis("off")
        del model; torch.cuda.empty_cache()

    figure.suptitle(f"Feature map mean activation: {dataset_name}")
    figure.tight_layout()
    figure.savefig(
        FIGURE_DIR / f"feature_maps_{dataset_name}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

# Section 2 — Light Fine-Tuning: Adapting the Encoder Without Forgetting

## Motivation

Section 1 trains only the linear head while keeping the encoder **completely frozen**. The project statement explicitly permits an alternative:

> *"Fine-tuning may be used, but it must preserve the encoder's ImageNet knowledge."*

This section documents — for each of the two encoders used in this project (`resnet50`, `convnext_tiny`) — **what light fine-tuning means in practice**, which parameters to unfreeze, which hyperparameters to use, and which mechanisms prevent the pretrained ImageNet representations from being overwritten.

No code is implemented here. This is the design reference; the implementation lives in Section 2 cells below.

---

## Why naive full fine-tuning destroys ImageNet knowledge

Unfreezing the entire encoder and training it with the head's learning rate (e.g. `1e-3`) reliably causes two failure modes on small target datasets:

**1. Large-gradient wash-out (catastrophic forgetting).**
At the start of training, the freshly initialized linear head produces near-random predictions and therefore a large loss. Back-propagating that loss at a high learning rate into a pretrained encoder overwrites carefully optimized ImageNet features before the head has had any chance to stabilize. This is the core mechanism of catastrophic forgetting in transfer learning [Kirkpatrick et al., 2017].

**2. Normalization-statistic drift.**
Both ResNet-50 and ConvNeXt-Tiny carry learned normalization terms — BatchNorm running mean/variance in ResNet, LayerNorm affine parameters in ConvNeXt — that were fit over 1.28 million ImageNet images. Re-estimating these on a small target set (e.g. Flowers-102 has ~1,000 training images; Stanford Cars ~8,000) degrades them toward a biased, high-variance estimate of a completely different distribution.

A "light" fine-tuning recipe must therefore address three things simultaneously:

- Keep the encoder learning rate **orders of magnitude smaller** than the head learning rate.
- **Restrict which parameters move** — preferably only the topmost blocks, closest to the task-specific semantics.
- **Protect normalization statistics** from being recomputed on the small target set.

---

## Shared principles (both encoders)

### Two-stage warm-up
The cleanest way to eliminate the initial random-gradient shock is the two-stage FitNets / transfer-learning strategy [Romero et al., 2015]:

1. **Phase 1:** Train only the head with the encoder completely frozen (identical to Section 1). Run this until the head converges or use the Phase 1 checkpoint directly.
2. **Phase 2:** Unfreeze the top encoder block(s) and continue training with a small encoder learning rate and the stable head from Phase 1.

Starting Phase 2 from a converged head means the gradients that flow into the encoder come from a reasonably trained classifier, not a random one. This dramatically reduces the initial gradient magnitude hitting the encoder.

### Discriminative / layer-wise learning rate decay (LLRD)
Different layers of a pretrained network capture representations at different levels of generality. Early layers (edges, textures, color blobs) are highly transferable across datasets; late layers (object parts, semantic categories) are more ImageNet-specific and need more adaptation for a new dataset. The appropriate tool is **layer-wise learning rate decay**:

```
lr_block = base_lr × decay^(depth_from_top)
```

with `decay ∈ [0.7, 0.9]`. The head gets the full `base_lr`; the top encoder block gets `base_lr × decay`; the next block down gets `base_lr × decay²`; and so on. Blocks kept frozen effectively have `lr = 0`. This formulation comes from ULMFiT [Howard & Ruder, 2018] and has been adopted in virtually every large-scale fine-tuning recipe since.

### Unfreeze only the top blocks
The knowledge-preserving choice in this project is to unfreeze **only the last 1–2 blocks** within the final encoder stage and keep everything below frozen. The final stage as a whole is 51–64% of the encoder's parameters, which is a large commitment; a single block is only ~17–20% (~4.5–4.8M params), giving much finer control over how much of the pretrained representation is allowed to move. These topmost blocks carry the most task-specific representations and are the most beneficial to adapt; the earlier blocks contain generic features that cost little to leave untouched.

### Short schedule and strong regularization
Fine-tuning runs should be significantly shorter than from-scratch training — typically 10–30 epochs. Cosine annealing, weight decay, and a small encoder learning rate (`1e-5`–`5e-5`) collectively prevent the encoder from drifting too far from its ImageNet initialization.

### Sanity metric
After fine-tuning, verify knowledge preservation quantitatively: compute the **relative encoder weight change** `‖Δθ_enc‖₂ / ‖θ_enc‖₂` before and after fine-tuning. A well-constrained run should show a small value (< 1–2%). As an additional check, frozen-encoder validation accuracy (from Phase 1) should not drop — if it does, the encoder has been overwritten.

---

## ResNet-50 specifics

### Architecture
ResNet-50 is composed of: `conv1 → bn1 → relu → maxpool → layer1 → layer2 → layer3 → layer4 → avgpool → fc`. For this project, the encoder is everything up to and excluding `avgpool`, so it ends at `layer4`. `layer4` contains 3 bottleneck blocks (~4.5–6.0M params each).

### What to unfreeze
For light fine-tuning, unfreeze the **last 1–2 bottleneck blocks of `layer4`** via `unfreeze_top(n_blocks=1)` (or `2`), keeping `conv1`, `bn1`, `layer1–3`, and the earlier blocks of `layer4` frozen. A single block is ~4.5M params (~19% of the encoder).

### The critical detail: BatchNorm running statistics
ResNet's pretrained knowledge is stored **not only in the convolutional weights** but also in the BatchNorm *running mean* and *running variance* buffers, which were accumulated over 1.28M ImageNet images. These buffers are not parameters (they don't appear in `model.parameters()`) and are therefore not updated by the optimizer — they are updated as a **side effect of the forward pass when the module is in training mode**.

To preserve them:

- **Keep the encoder's `.eval()` mode even while unfreezing parameters.** This makes every BatchNorm layer use its stored running stats rather than recomputing them from the current batch. `TeacherModel.train()` already forces `self.encoder.eval()` — this behavior must be maintained even after the top blocks of `layer4` are unfrozen. Critically, setting `requires_grad=True` on a parameter (enabling gradient flow) is **independent** of the module's `training` flag (controlling BN statistic updates): the two axes are orthogonal.
- **Freeze the BN affine parameters** (`weight` and `bias`) inside the unfrozen blocks as well. This is the "frozen BN" recipe standard in detection and segmentation transfer (e.g. Faster R-CNN, Mask R-CNN). `unfreeze_top` does this automatically for ResNet-50.

### AMP / GradScaler note
The frozen-encoder training in Section 1 skips `GradScaler` because `loss.backward()` only computes gradients for the tiny linear head — the scale overflow risk is negligible. Once encoder blocks carry gradients (millions of parameters), **re-enable `GradScaler`** for numerically stable mixed-precision training.

### Recipe summary

| Setting | Value |
|---|---|
| Start from | Phase 1 checkpoint (converged head) |
| Unfrozen encoder params | last N bottleneck blocks of `layer4` (default: 1 block, ~4.5M params) |
| BatchNorm mode | `.eval()` throughout (frozen running stats) |
| Head LR | `1e-3` |
| Encoder LR (top block) | `1e-4` to `1e-5` (LLRD `decay ≈ 0.9`) |
| Optimizer | AdamW |
| Schedule | Cosine annealing |
| Epochs (Phase 2) | 10–20 |
| Weight decay | `1e-4` |
| GradScaler | Enabled |

---

## ConvNeXt-Tiny specifics

### Architecture
ConvNeXt-Tiny's `model.features` is a sequence of 8 blocks indexed 0–7: a stem at index 0, then alternating downsampling layers and stages at indices 1–7, giving four stages with depths `[3, 3, 9, 3]`. The final stage is `features[7]` (768-dim, 7×7) and contains 3 CNBlocks of ~4.76M params each.

### What to unfreeze
For light fine-tuning, unfreeze the **last 1–2 CNBlocks of `features[7]`** via `unfreeze_top(n_blocks=1)` (or `2`), keeping the stem, the first three stages, and the earlier blocks of `features[7]` frozen. A single CNBlock is ~4.76M params (~17% of the encoder).

### No BatchNorm — a key difference from ResNet-50
ConvNeXt uses **LayerNorm** rather than BatchNorm everywhere. Unlike BatchNorm, LayerNorm computes statistics per-sample at runtime (no running buffers to corrupt) and is safe to use in training mode on small datasets. This makes ConvNeXt **more forgiving** than ResNet for fine-tuning: there is no need for the frozen-BN workaround, and no separate `.eval()` mode enforcement is required for the normalization layers.

### Official fine-tuning recipe
The ConvNeXt paper [Liu et al., CVPR 2022] provides a specific fine-tuning recipe for downstream tasks. The key settings are:

- **AdamW optimizer** with a base learning rate of **`5e-5`** (two orders of magnitude below the from-scratch LR of `4e-3`).
- **Layer-wise LR decay** with `decay ≈ 0.8` — the official codebase calls this `--layer_decay 0.8`. Each block deeper in the network gets its LR multiplied by 0.8 relative to the block above it.
- **Stochastic depth** (`drop_path`): ConvNeXt applies stochastic depth as a regularizer during fine-tuning with `drop_path ≈ 0.1–0.2`. The torchvision ConvNeXt blocks already contain `stochastic_depth` submodules; setting their drop probability > 0 randomly drops entire residual branches during each forward pass, preventing the unfrozen blocks from drifting too aggressively. The fvcore FLOP-counting warning we observed earlier about `stochastic_depth` submodules is a direct consequence of this architecture.
- **LayerScale**: ConvNeXt initializes each residual branch with a learnable per-channel scale initialized to `1e-6`. This ensures residual updates start extremely small, providing a natural guard against large initial perturbations during fine-tuning.
- **Label smoothing** (`0.1`): standard regularization in the ConvNeXt recipe; helps prevent overconfident updates on small datasets.

### Recipe summary

| Setting | Value |
|---|---|
| Start from | Phase 1 checkpoint (converged head) |
| Unfrozen encoder params | last N CNBlocks of `features[7]` (default: 1 block, ~4.76M params) |
| LayerNorm mode | Training mode is safe (per-sample stats, no buffers) |
| Head LR | `1e-3` |
| Encoder LR (top block) | `5e-5` (LLRD `decay ≈ 0.8`) |
| Optimizer | AdamW |
| Stochastic depth | `drop_path ≈ 0.1–0.2` |
| LayerScale init | `1e-6` (already in torchvision weights) |
| Label smoothing | `0.1` |
| Schedule | Cosine annealing |
| Epochs (Phase 2) | 10–30 |
| Weight decay | `1e-4` |
| GradScaler | Recommended (encoder gradients now flow) |

---

## How this plugs into the existing codebase

The frozen-encoder baseline from Section 1 remains reproducible and unchanged. The fine-tuning variant is additive:

1. **`TeacherModel.unfreeze_top(n_blocks=1)`:** Unfreezes the last `n_blocks` sub-blocks within the final encoder stage by calling `requires_grad_(True)` on them, while keeping `TeacherModel.train()`'s `encoder.eval()` enforcement intact (matters for ResNet BN). Unfreezing a parameter and setting eval mode are orthogonal — they can be set independently. For ResNet-50 the method re-freezes BatchNorm affine params inside the unfrozen blocks. Block-level unfreezing is not implemented for VGG-16-BN (its encoder is a flat `nn.Sequential` with no block containers).

2. **`train_teacher_finetune(...)` in `train.py`:** A variant of `train_teacher_classifier` that builds **parameter groups** for AdamW — one group per unfrozen encoder block with LLRD-scaled LRs (topmost block at `encoder_lr`, each deeper block × `lr_decay`), plus the head group at full `lr` — and enables `GradScaler`. The optimizer construction is the main difference; the training loop itself is identical.

3. **Separate checkpoint directory:** Fine-tuned checkpoints are saved to `checkpoints/teachers/finetune/{teacher}_{dataset}.pth` (frozen baselines live in `checkpoints/teachers/frozen/`), so the two coexist and can be compared side-by-side in the results table. The fine-tuned checkpoint also stores `encoder_state_dict` and `n_blocks_unfrozen`.

---

## References

- [1] Liu, Z. et al. *A ConvNet for the 2020s* (ConvNeXt). CVPR 2022. https://arxiv.org/abs/2201.03545
- [2] He, K. et al. *Deep Residual Learning for Image Recognition* (ResNet). CVPR 2016. https://arxiv.org/abs/1512.03385
- [3] Howard, J. & Ruder, S. *Universal Language Model Fine-Tuning for Text Classification* (ULMFiT / LLRD). ACL 2018. https://arxiv.org/abs/1801.06146
- [4] Hinton, G. et al. *Distilling the Knowledge in a Neural Network*. NeurIPS Workshop 2015. https://arxiv.org/abs/1503.02531
- [5] Romero, A. et al. *FitNets: Hints for Thin Deep Nets*. ICLR 2015. https://arxiv.org/abs/1412.6550
- [6] Kirkpatrick, J. et al. *Overcoming Catastrophic Forgetting in Neural Networks* (EWC). PNAS 2017. https://arxiv.org/abs/1612.00796
- ConvNeXt official fine-tuning configs (layer_decay 0.8, drop_path 0.2): https://deepwiki.com/facebookresearch/ConvNeXt/5.2-training-configurations


In [ ]:
# Fine-tuning uses the same TEACHERS / DATASETS / NUM_WORKERS from Section 1.
# Add entries to RUN_OVERRIDES_FT for any run that needs non-default settings.
# n_blocks: number of blocks to unfreeze within the final encoder stage
#   convnext_tiny: 1 block ≈ 4.84M params (17%), 2 blocks ≈ 9.60M (34%), 3 = full stage
#   resnet50:      1 block ≈ 4.66M params (20%), 2 blocks ≈ 8.93M (38%), 3 = full layer4
DEFAULTS_FT = dict(
    num_epochs=2,
    lr=1e-3,
    encoder_lr=5e-5,
    lr_decay=0.8,
    weight_decay=1e-4,
    batch_size=256,
    label_smoothing=0.1,
    n_blocks=1,
)


In [ ]:
histories_ft = {}
history_frames_ft = []

for teacher_name in TEACHERS:
    for dataset_name in DATASETS:
        cfg = {**DEFAULTS_FT, **RUN_OVERRIDES_FT.get((teacher_name, dataset_name), {})}

        print("=" * 100)
        print(f"[FT] Teacher: {teacher_name} | Dataset: {dataset_name} | cfg: {cfg}")
        set_seed(291652)

        train_loader, val_loader, _, num_classes = get_dataloaders(
            dataset_name=dataset_name,
            data_root=DATA_ROOT,
            batch_size=cfg["batch_size"],
            num_workers=NUM_WORKERS,
        )
        model = TeacherModel(backbone_name=teacher_name, num_classes=num_classes)

        # Warm-start: load the frozen Phase 1 head before unfreezing the encoder.
        frozen_ckpt = checkpoint_path(teacher_name, dataset_name, mode="frozen")
        if frozen_ckpt.exists():
            ckpt = torch.load(frozen_ckpt, map_location="cpu")
            model.classifier.load_state_dict(ckpt["classifier_state_dict"])
            print(f"  Loaded frozen head from {frozen_ckpt.name} "
                  f"(best val acc: {ckpt.get('best_val_acc', 'n/a'):.2f}%)")
        else:
            print(f"  WARNING: frozen checkpoint not found at {frozen_ckpt}. "
                  "Starting from a random head.")

        model.unfreeze_top(cfg["n_blocks"])
        describe_model(model, train_loader)

        history = train_teacher_finetune(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=cfg["num_epochs"],
            lr=cfg["lr"],
            encoder_lr=cfg["encoder_lr"],
            lr_decay=cfg["lr_decay"],
            weight_decay=cfg["weight_decay"],
            label_smoothing=cfg["label_smoothing"],
            n_blocks=cfg["n_blocks"],
            device=DEVICE,
            checkpoint_path=str(checkpoint_path(teacher_name, dataset_name, mode="finetune")),
        )
        histories_ft[(teacher_name, dataset_name)] = history

        history_frame = save_training_history_csv(
            history=history,
            save_path=history_csv_path(teacher_name, dataset_name, mode="finetune"),
            teacher_name=teacher_name,
            dataset_name=dataset_name,
        )
        history_frames_ft.append(history_frame)
        pd.concat(history_frames_ft, ignore_index=True).to_csv(
            TABLE_DIR / "teacher_training_histories_finetune.csv",
            index=False,
        )

        plot_training_curves(
            history=history,
            title=f"{teacher_name} on {dataset_name} (fine-tuned)",
            save_path=FIGURE_DIR / f"teacher_finetune_{teacher_name}_{dataset_name}.png",
        )
        plt.show()
        del model; torch.cuda.empty_cache()

In [ ]:
results_ft = []
flop_cache_ft = {}

for teacher_name in TEACHERS:
    for dataset_name in DATASETS:
        ckpt_path = checkpoint_path(teacher_name, dataset_name, mode="finetune")
        if not ckpt_path.exists():
            print(f"Skipping missing checkpoint: {ckpt_path}")
            continue

        _, _, test_loader, num_classes = get_dataloaders(
            dataset_name=dataset_name,
            data_root=DATA_ROOT,
            batch_size=DEFAULTS_FT["batch_size"],
            num_workers=NUM_WORKERS,
        )
        model = TeacherModel(backbone_name=teacher_name, num_classes=num_classes).to(DEVICE)
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model.classifier.load_state_dict(checkpoint["classifier_state_dict"])
        # Restore the fine-tuned encoder weights.
        model.encoder.load_state_dict(checkpoint["encoder_state_dict"])

        _, test_acc = evaluate(model, test_loader, DEVICE)
        flop_key = (teacher_name, num_classes)
        if flop_key not in flop_cache_ft:
            try:
                flop_cache_ft[flop_key] = compute_flops(model)
            except Exception as exc:
                print(f"Could not compute FLOPs: {exc}")
                flop_cache_ft[flop_key] = np.nan

        results_ft.append(
            {
                "Mode": "finetune",
                "Teacher": teacher_name,
                "Dataset": dataset_name,
                "Val Acc (%)": checkpoint.get("best_val_acc", np.nan),
                "Test Acc (%)": test_acc,
                "Total Params (M)": count_total_parameters(model) / 1e6,
                "Trainable Params": count_parameters(model),
                "GFLOPs": flop_cache_ft[flop_key],
                "Blocks Unfrozen": checkpoint.get("n_blocks_unfrozen", 1),
            }
        )
        del model; torch.cuda.empty_cache()

results_ft_df = pd.DataFrame(results_ft)
display(results_ft_df)
results_ft_df.to_csv(TABLE_DIR / "teacher_results_finetune.csv", index=False)

In [ ]:
# Side-by-side comparison: frozen linear probe vs. fine-tuned encoder.
frozen_csv = TABLE_DIR / "teacher_results.csv"
finetune_csv = TABLE_DIR / "teacher_results_finetune.csv"

frames = []
if frozen_csv.exists():
    df_frozen = pd.read_csv(frozen_csv)
    df_frozen.insert(0, "Mode", "frozen")
    df_frozen["Blocks Unfrozen"] = 0
    frames.append(df_frozen)
if finetune_csv.exists():
    frames.append(pd.read_csv(finetune_csv))

if frames:
    comparison_df = (
        pd.concat(frames, ignore_index=True)
        .sort_values(["Teacher", "Dataset", "Mode"])
        .reset_index(drop=True)
    )
    display(comparison_df)
    comparison_df.to_csv(TABLE_DIR / "teacher_results_comparison.csv", index=False)
    print("Saved → results/tables/teacher_results_comparison.csv")
else:
    print("No result CSVs found — run the training cells first.")